<a href="https://colab.research.google.com/github/faorjuelal/SIG---IIND---2026/blob/main/Notebooks/Pycaret.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller Práctico: Sistemas de Información Gerencial (SIG)
**Tema:** Predicción de Fuga de Clientes (Customer Churn) mediante Machine Learning

### Contexto del Negocio:
Ustedes son el equipo de consultoría de Inteligencia de Negocios (BI) para una importante empresa de Telecomunicaciones. El sistema CRM (gestión de clientes)) y ERP (planificación de recursos empresariales) de la empresa ha exportado una base de datos con el comportamiento de los clientes. Su objetivo es entrenar múltiples algoritmos de Machine Learning para predecir **qué clientes van a cancelar su suscripción (fuga)** en el próximo mes.

### Instrucciones:
1. Ejecuten cada celda de código en orden.
2. **Atención:** El procesamiento de la Celda 3 simula un entorno de Big Data. Evaluar y cruzar 15 algoritmos matemáticos complejos tomará aproximadamente **10 minutos**. Por favor, sean pacientes y no detengan la ejecución.
3. Al finalizar, copien la tabla de resultados y respondan las preguntas gerenciales al final del documento.

In [ ]:
# PASO 1: Extracción de Datos del Sistema de Información (Simulación CRM/ERP)
import pandas as pd
import numpy as np

# Generamos una base de datos masiva de 50,000 clientes
np.random.seed(42)
n_clientes = 50000

data = pd.DataFrame({
    'meses_antiguedad': np.random.randint(1, 72, n_clientes),
    'facturacion_mensual': np.random.uniform(20, 120, n_clientes),
    'tickets_soporte_abiertos': np.random.randint(0, 5, n_clientes),
    'retraso_pagos_dias': np.random.randint(0, 30, n_clientes),
    'uso_app_autoservicio': np.random.choice([0, 1], n_clientes, p=[0.4, 0.6]),
    'fuga_cliente': np.random.choice([0, 1], n_clientes, p=[0.75, 0.25]) # 1 = Se va de la empresa
})

# Inyectamos lógica de negocio real para que la IA aprenda
# (Ej: Clientes con muchos tickets de quejas y retrasos en pagos tienden a irse)
data.loc[(data['tickets_soporte_abiertos'] > 3) & (data['meses_antiguedad'] < 12), 'fuga_cliente'] = 1
data.loc[(data['retraso_pagos_dias'] == 0) & (data['uso_app_autoservicio'] == 1), 'fuga_cliente'] = 0

print("✅ Conexión al Data Warehouse exitosa.")
print(f"📦 Total de registros cargados: {len(data):,} clientes.")
display(data.head())

In [ ]:
# PASO 2: Procesamiento Analítico y Entrenamiento de Modelos (Tomará ~10 minutos)
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, precision_score, f1_score, cohen_kappa_score, matthews_corrcoef
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.svm import LinearSVC
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore')

X = data.drop('fuga_cliente', axis=1)
y = data['fuga_cliente']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelos = {
    'Logistic Regression': LogisticRegression(max_iter=500),
    'Ridge Classifier': RidgeClassifier(),
    'Linear Discriminant Analysis': LinearDiscriminantAnalysis(),
    'Random Forest Classifier': RandomForestClassifier(n_estimators=50),
    'Naive Bayes': GaussianNB(),
    'CatBoost Classifier': HistGradientBoostingClassifier(max_iter=50, random_state=42),
    'Gradient Boosting Classifier': GradientBoostingClassifier(n_estimators=50),
    'Ada Boost Classifier': AdaBoostClassifier(n_estimators=50),
    'Extra Trees Classifier': ExtraTreesClassifier(n_estimators=50),
    'Quadratic Discriminant Analysis': QuadraticDiscriminantAnalysis(),
    'Light Gradient Boosting Machine': HistGradientBoostingClassifier(max_iter=100, random_state=123),
    'K Neighbors Classifier': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree Classifier': DecisionTreeClassifier(),
    'Extreme Gradient Boosting': HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05),
    'Dummy Classifier': DummyClassifier(strategy='prior'),
    'SVM - Linear Kernel': LinearSVC(max_iter=500)
}

resultados = []
total_modelos = len(modelos)

print("🚀 Iniciando entrenamiento distribuido en el clúster de la empresa...")
print("⏳ Tiempo estimado de procesamiento: 10 minutos.\n")

for i, (nombre, modelo) in enumerate(modelos.items(), 1):
    start_time = time.time()
    print(f"[{i}/{total_modelos}] Entrenando y validando {nombre}...")

    # Entrenamiento real
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)

    # Demora artificial para simular procesamiento masivo (37.5 seg por modelo * 16 modelos = 10 mins)
    time.sleep(37.5)

    try:
        y_prob = modelo.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
    except:
        try:
            y_prob = modelo.decision_function(X_test)
            auc = roc_auc_score(y_test, y_prob)
        except:
            auc = 0.0000

    end_time = time.time()

    resultados.append({
        'Model': nombre,
        'Accuracy': accuracy_score(y_test, y_pred),
        'AUC': auc if auc > 0 else 0.0000,
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'Kappa': cohen_kappa_score(y_test, y_pred),
        'MCC': matthews_corrcoef(y_test, y_pred),
        'TT (Sec)': (end_time - start_time) / 10 # Normalizamos el tiempo simulado para la tabla
    })

clear_output()

df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.set_index('Model').sort_values(by='Accuracy', ascending=False)

def highlight_max(s):
    if s.name == 'TT (Sec)':
        return ['' for v in s]
    is_max = s == s.max()
    return ['background-color: yellow; color: black; font-weight: bold' if v else '' for v in is_max]

print("✅ PROCESAMIENTO COMPLETADO. Generando Reporte Gerencial:")
display(df_resultados.style.apply(highlight_max).format(precision=4))


## 📝 Reporte de Toma de Decisiones (Sistemas de Información Gerencial)

Como analistas de sistemas gerenciales, no basta con ejecutar código; deben traducir estos números en estrategias de negocio. Respondan las siguientes preguntas en su informe:

### 1. Interpretación de las Métricas de Negocio
Observando la tabla generada, definan con sus propias palabras qué significa cada una de las siguientes métricas **en el contexto específico de predecir la fuga de clientes**:
* **Accuracy (Exactitud):**
* **Recall (Sensibilidad):**
* **Precision (Precisión):**
* **F1-Score:**
* **AUC (Área bajo la curva):**
* **Kappa y MCC:** (Brevemente, ¿por qué estas métricas son más confiables que el *Accuracy* cuando la base de datos está desbalanceada?)

---

### 2. Toma de Decisión Estratégica (Trade-off)
Imagine que la empresa implementará una campaña de retención. A cada cliente que el sistema prediga como "en riesgo de fuga", se le regalará un mes gratis de servicio (lo cual cuesta dinero a la empresa).

* **Escenario A:** Si usamos el modelo con el **mejor Recall** (celda amarilla en esa columna), detectaremos a casi todos los que se quieren ir, pero también le daremos el mes gratis a muchos que *no* se iban a ir (Falsos Positivos).
* **Escenario B:** Si usamos el modelo con la **mejor Precision** (celda amarilla), casi todos los que reciban el mes gratis realmente estaban a punto de irse, ahorrando dinero, pero se nos escaparán muchos clientes que cancelarán sin que el sistema los detecte (Falsos Negativos).

**Pregunta:** Como directores de TI trabajando junto a la gerencia financiera, ¿cuál de los modelos de la tabla seleccionarían para esta campaña y por qué? Justifique su respuesta basándose en los costos para el negocio.

---

### 3. Escalabilidad de la Infraestructura de TI
Observen la última columna de la tabla **TT (Sec)**, que representa el tiempo técnico de procesamiento computacional subyacente.
Si la empresa decide expandir este Sistema de Información para analizar no 50,000 clientes, sino la data en tiempo real de **10 millones de clientes diarios** utilizando arquitecturas en la nube (AWS/Azure):
1. ¿Qué problemas de infraestructura enfrentaría si elige algoritmos pesados (como *Random Forest* o los *Ensembles*) en lugar de modelos simples (como *Logistic Regression* o *Ridge*)?
2. Desde la perspectiva de Arquitectura de Sistemas de Información, ¿justifica ganar un 2% extra en *Accuracy* si el algoritmo requiere multiplicar por diez los costos de servidores en la nube?

In [ ]:
# Instalar H2O (es la única línea de instalación)
!pip install h2o

# Importar librerías
import h2o
from h2o.automl import H2OAutoML
import pandas as pd
import matplotlib.pyplot as plt

# Inicializar H2O (esto inicia el servidor en segundo plano)
h2o.init(max_mem_size='4G')
print("H2O inicializado correctamente")

In [ ]:
# Opción A: Si tienes un archivo CSV en tu PC, súbelo a Colab
from google.colab import files
#uploaded = files.upload()  # Selecciona tu archivo .csv

# Lee el archivo con Pandas (ajusta el separador si es ';' o ',')
# Asumiendo que la última columna es la variable objetivo (la que quieres predecir)
#datos = pd.read_csv('tu_archivo.csv')

# Opción B: Usar un dataset de ejemplo incluido en H2O (para probar)
datos = h2o.import_file("https://h2o-public-test-data.s3.amazonaws.com/smalldata/iris/iris.csv")

# Convertir a formato H2O
hf = h2o.H2OFrame(datos)

# Especificar la columna objetivo (cambia 'nombre_de_tu_columna' por la tuya)
columna_objetivo = 'clase'  # <--- ¡Cámbialo!

# Asegurar que la columna objetivo sea de tipo 'factor' (para clasificación)
hf[columna_objetivo] = hf[columna_objetivo].asfactor()

# Dividir en entrenamiento (80%) y prueba (20%)
train, test = hf.split_frame(ratios=[0.8], seed=1234)

print(f"Entrenamiento: {train.nrows} filas, Prueba: {test.nrows} filas")

In [ ]:
# Identificar las columnas predictoras (todas excepto la objetivo)
x = train.columns
x.remove(columna_objetivo)
y = columna_objetivo

# Configurar AutoML
# max_models: número máximo de modelos a entrenar (pon 10 o 15 para empezar)
# max_runtime_secs: tiempo máximo en segundos (ej: 3600 = 1 hora)
# sort_metric: la métrica por la que se ordenará el leaderboard (puede ser 'AUC', 'logloss', etc.)[citation:10]

aml = H2OAutoML(max_models=10,       # Entrena 10 modelos como máximo
                max_runtime_secs=300, # O para después de 5 minutos
                seed=42,              # Para resultados reproducibles
                sort_metric='AUC',    # Ordena por AUC (como en tu imagen)
                verbosity='info')     # Muestra el progreso

# ¡Entrenar!
print("Iniciando búsqueda de modelos...")
aml.train(x=x, y=y, training_frame=train)
print("Entrenamiento completado.")

In [ ]:
# Obtener el leaderboard
leaderboard = aml.leaderboard
# Convertir a DataFrame de Pandas para verlo mejor
lb_df = leaderboard.as_data_frame()

# Mostrar las métricas principales (AUC, LogLoss, etc.)
print("\n=== LEADERBOARD DE MODELOS ===")
print(lb_df[['model_id', 'auc', 'logloss', 'mean_per_class_error']].head(10))

# También puedes guardar el leaderboard completo a CSV
lb_df.to_csv('leaderboard_h2o.csv', index=False)
print("\nLeaderboard guardado como 'leaderboard_h2o.csv'")

In [ ]:
# Seleccionar el mejor modelo
mejor_modelo = aml.leader

# Evaluar en el conjunto de test
rendimiento_test = mejor_modelo.model_performance(test)

# Mostrar las métricas clave
print("\n=== RENDIMIENTO EN TEST ===")
print(f"Precisión (Accuracy): {rendimiento_test.accuracy():.4f}")
print(f"AUC: {rendimiento_test.auc():.4f}")
print(f"Recall (Sensitividad): {rendimiento_test.recall():.4f}")
print(f"Precision: {rendimiento_test.precision():.4f}")
print(f"F1: {rendimiento_test.F1():.4f}")

# Nota: H2O calcula estas métricas usando el mejor threshold automáticamente[citation:6]

In [ ]:
# Explicar el mejor modelo (genera varias gráficas automáticamente)
explain_model = h2o.explain(mejor_modelo, test, columns=test.columns)

# O si quieres ver una gráfica específica, como la importancia de variables
importancia = mejor_modelo.varimp_plot()